# Appendix — Probe training curves

Tensorboard logs of the probe trainings (`fit_unsloth.py`): Fig 10 of the paper, plus extra diagnostics (correlations and Δr along training, run screening, end-of-training table). Curves: mean ± SEM over runs, red = end probe, orange = mid probe.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
from notebook_helper import *

logs = load_training_logs()   # logs[dataset][model][probe] -> [(run name, scalars), ...]

In [ ]:
def interp_stack(curves, n_grid=200):
    """(grid, mean, sem, n_runs) of (steps, values) curves, on the step range they all cover."""
    lo, hi = max(x[0] for x, _ in curves), min(x[-1] for x, _ in curves)
    if not hi > lo:
        return None
    n = min(n_grid, min(len(x) for x, _ in curves))    # no denser than the sparsest run
    grid = np.linspace(lo, hi, max(n, 2))
    stack = np.vstack([np.interp(grid, x, v) for x, v in curves])
    err = stats.sem(stack, axis=0) if len(stack) > 1 else np.zeros_like(grid)
    return grid, stack.mean(axis=0), err, len(stack)


def mean_sem(d, m, p, key):
    curves = [s for s in (get_series(sc, key) for _, sc in logs[d][m][p])
              if s is not None and len(s[0]) > 1]
    return interp_stack(curves) if curves else None


def delta_mean_sem(d, m, p):
    """Δr = r(conf, accuracy) − r(conf, consistency), paired per run before averaging."""
    curves = []
    for _, sc in logs[d][m][p]:
        a, c = get_series(sc, "test_corr_pred_acc"), get_series(sc, "test_corr_pred_cons")
        if a is not None and c is not None and len(a[0]) >= 2:
            curves.append((a[0], a[1] - np.interp(a[0], c[0], c[1])))
    return interp_stack(curves) if curves else None


def final_values(d, m, p, key):
    """Last logged value of one tag, one per run."""
    series = (get_series(sc, key) for _, sc in logs[d][m][p])
    return np.array([s[1][-1] for s in series if s is not None and len(s[1])], dtype=float)

In [ ]:
FS = dict(base=8, title=9, label=9, tick=7, legend=8)
set_style(FS, lw=0.6)
DEV_LABELS = {"end": "end probe", "mid": "mid probe"}
SHORT = [SHORT_LABELS[d] for d in DATASETS]


def metric_grid(series, ylabel, name, logy=False, ylim=None, zero_line=False,
                baseline_key=None, baseline_label=None, curve_fn=mean_sem):
    """Rows = models, columns = datasets; one mean ± SEM band per (probe, series).

    series: [(key, linestyle, label), ...], key passed to curve_fn(d, m, p, key).
    baseline_key: tag drawn as a flat reference line (e.g. r(consistency, accuracy)).
    """
    fig, axs = plt.subplots(len(MODELS), len(DATASETS),
                            figsize=(1.5 * len(DATASETS) + 2.7, 0.8 * len(MODELS) + 0.9),
                            sharex="col", sharey="row" if logy else True, layout="constrained")
    fig.get_layout_engine().set(w_pad=0.01, h_pad=0.01, wspace=0.012, hspace=0.012)
    n_min = np.inf
    for i, model in enumerate(MODELS):
        for j, d in enumerate(DATASETS):
            ax = axs[i, j]
            drawn = 0
            for probe in PROBES:
                for key, ls, _ in series:
                    res = curve_fn(d, model, probe, key)
                    if res is None:
                        continue
                    x, mean, err, n_used = res
                    drawn, n_min = max(drawn, n_used), min(n_min, n_used)
                    ax.fill_between(x, mean - err, mean + err, color=COLORS[probe], alpha=0.3,
                                    lw=0, zorder=3)
                    ax.plot(x, mean, color=COLORS[probe], ls=ls, lw=1.0, zorder=4)
            if baseline_key is not None:
                vals = np.concatenate([final_values(d, model, p, baseline_key) for p in PROBES])
                vals = vals[~np.isnan(vals)]
                if len(vals):
                    ax.axhline(vals.mean(), color=COLORS["navy"], ls=":", lw=0.9, zorder=2)

            if drawn == 0:
                ax.text(0.5, 0.5, "no runs", ha="center", va="center", transform=ax.transAxes,
                        color="0.6", fontsize=FS["tick"])
                ax.set_xticks([])
            elif drawn < N_RUNS:
                ax.text(0.97, 0.92, f"n={drawn}", ha="right", va="top", transform=ax.transAxes,
                        color="0.45", fontsize=FS["tick"] - 1)
            if zero_line:
                ax.axhline(0, color="black", lw=0.6, zorder=2)
            if logy:
                ax.set_yscale("log")
                ax.yaxis.set_major_locator(plt.LogLocator(numticks=4))
                ax.yaxis.set_minor_locator(plt.NullLocator())
            else:
                ax.yaxis.set_major_locator(plt.MaxNLocator(3))
            if ylim is not None:
                ax.set_ylim(*ylim)
            style_ax(ax, color="0.92", lw=0.5)
            ax.margins(x=0.02)
            ax.tick_params(length=2, pad=1.5)
            ax.xaxis.set_major_locator(plt.MaxNLocator(3, integer=True))
            if j:
                ax.tick_params(axis="y", length=0)
            if i == 0:
                ax.set_title(SHORT[j], pad=3)
            if j == 0:
                ax.set_ylabel(model, rotation=0, ha="right", va="center", fontsize=FS["tick"],
                              labelpad=6)

    handles = [Line2D([], [], color=COLORS[p], lw=1.6, label=DEV_LABELS[p]) for p in PROBES]
    handles += [Line2D([], [], color="0.35", lw=1.6, ls=ls, label=label)
                for _, ls, label in series if label]
    if baseline_key is not None and baseline_label:
        handles.append(Line2D([], [], color=COLORS["navy"], ls=":", lw=1.2, label=baseline_label))
    n_band = int(n_min) if np.isfinite(n_min) else N_RUNS
    handles.append(Line2D([], [], color="0.6", lw=6, alpha=0.3, label=f"± SEM over runs (≥{n_band})"))
    fig.legend(handles=handles, loc="outside upper center", ncols=len(handles), frameon=False,
               handlelength=1.8, columnspacing=1.4, borderaxespad=0.1)
    fig.supxlabel("training step", fontsize=FS["label"])
    fig.supylabel(ylabel, fontsize=FS["label"])
    save_fig(fig, name)
    plt.show()

## Fig 10 — Loss along training
Held-out subset (solid) and training subset (dashed), log scale.

In [ ]:
metric_grid([("eval_test_loss", "-", "held-out subset"),
             ("eval_train_loss", "--", "training subset")],
            ylabel="probe MSE loss (log scale)", name="fig10_loss", logy=True)

## Extra — Correlations along training
Held-out subset. Dotted navy line: r(consistency, accuracy), a property of the data.

In [ ]:
metric_grid([("test_corr_pred_acc", "-", "r(conf, accuracy)"),
             ("test_corr_pred_cons", "--", "r(conf, consistency)")],
            ylabel="Pearson r (held-out subset)", name="extra_training_correlations", ylim=(-0.1, 1.0),
            zero_line=True, baseline_key="test_corr_cons_acc",
            baseline_label="r(consistency, accuracy)")

## Extra — Δr along training
Δr = r(conf, accuracy) − r(conf, consistency), paired per run before averaging.

In [ ]:
metric_grid([(None, "-", None)], ylabel="Δ Pearson r (accuracy − consistency)",
            name="extra_training_delta", zero_line=True, curve_fn=lambda d, m, p, key: delta_mean_sem(d, m, p))

## Run screening
Held-out loss with NaN, diverging (final > initial) or above 2x the median of its configuration.

In [ ]:
OUT_FACTOR = 2.0
suspects = []
for d in DATASETS:
    for m in MODELS:
        for p in PROBES:
            losses = [(run, get_series(sc, "eval_test_loss")) for run, sc in logs[d][m][p]]
            losses = [(run, s[1]) for run, s in losses if s is not None and len(s[1])]
            if not losses:
                continue
            med = np.nanmedian([v[-1] for _, v in losses])
            for run, v in losses:
                reasons = []
                if np.isnan(v).any():
                    reasons.append("NaN in loss")
                if len(v) > 1 and v[-1] > v[0]:
                    reasons.append(f"diverged ({v[0]:.4f} -> {v[-1]:.4f})")
                if np.isfinite(med) and med > 0 and v[-1] > OUT_FACTOR * med:
                    reasons.append(f"final {v[-1]:.4f} > {OUT_FACTOR}x median ({med:.4f})")
                if reasons:
                    suspects.append({"dataset": SHORT_LABELS[d], "model": m, "probe": p,
                                     "run": run, "reason": "; ".join(reasons)})

pd.DataFrame(suspects) if suspects else print("no pathological run")

## End-of-training table
Last eval step of every configuration, mean ± SEM over runs.

In [ ]:
def fmt(v, sign=""):
    return f"{v.mean():{sign}.3f} ± {stats.sem(v):.3f}" if len(v) > 1 else f"{v.mean():{sign}.3f}"


rows = []
for d in DATASETS:
    for m in MODELS:
        for p in PROBES:
            if not logs[d][m][p]:
                continue
            row = {"dataset": SHORT_LABELS[d], "model": m, "probe": p, "n_runs": len(logs[d][m][p])}
            for key, col in [("eval_test_loss", "test loss"), ("test_pass005", "pass@0.05"),
                             ("test_corr_pred_acc", "r(conf,acc)"),
                             ("test_corr_pred_cons", "r(conf,cons)"),
                             ("test_corr_cons_acc", "r(cons,acc)")]:
                v = final_values(d, m, p, key)
                v = v[~np.isnan(v)]
                if len(v):
                    row[col] = fmt(v)
            a = final_values(d, m, p, "test_corr_pred_acc")
            c = final_values(d, m, p, "test_corr_pred_cons")
            if len(a) and len(a) == len(c):
                row["Δr"] = fmt(a - c, sign="+")
            rows.append(row)

summary = pd.DataFrame(rows)
os.makedirs(FIG_PATH, exist_ok=True)
summary.to_csv(os.path.join(FIG_PATH, "table_end_of_training.csv"), index=False)
pd.set_option("display.max_rows", None)
summary